# 05b — Validation before model tuning

**Purpose:** confirm launch-time assumptions and test whether baseline model conclusions depend heavily on questionable feature groups or encoding. This is a **training-only** review after `05_modeling_v3.ipynb` and before `06_evaluation_and_tuning.ipynb`.

**Important:** This notebook does **not open the saved test IDs, `X_test`, or `y_test`**, select a final model, or compute final performance. It requires the local, gitignored outputs of `02_clean_data.ipynb` and `03_preprocess_data.ipynb`. It preserves the shared `utils/features.py` implementation unchanged. See `docs/model_validation.md`.

The meaning of `funded date`, the historical availability of the scraped title/rewards, and the geographic scope must still be confirmed from original source information. Numerical plausibility is not documentary proof.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, roc_auc_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.features import add_base_columns
from utils.preprocessing import FeatureBuilder, drop_leaks, make_pipeline

SEED = 42
pd.set_option("display.width", 150)

## 1. Restore the official *training* split

This reconstruction reads only saved **train** IDs from notebook `03`. The target is checked against cleaned campaign status; no second random split is performed. Later comparisons use stratified folds **inside** this training partition.

In [2]:
processed = PROJECT_ROOT / "data" / "Processed-Data"
clean_path = processed / "DSI_kickstarterscrape_dataset_cleaned.csv"
train_ids_path = processed / "train_project_ids.csv"
for required in (clean_path, train_ids_path):
    if not required.is_file():
        raise FileNotFoundError(f"Missing {required}; run notebooks 02 and 03 first")

clean = pd.read_csv(clean_path)
train_ids = pd.read_csv(train_ids_path)
assert clean["project id"].is_unique, "Cleaned project IDs must be unique"
assert train_ids["project id"].is_unique, "Training project IDs must be unique"

rows = clean.set_index("project id", drop=False).loc[train_ids["project id"]].copy()
y_train = train_ids["target"].astype(int).reset_index(drop=True)
assert np.array_equal(rows["status"].eq("successful").astype(int).to_numpy(), y_train.to_numpy())

X_train = drop_leaks(rows.drop(columns=["status"]).reset_index(drop=True))
assert X_train["project id"].tolist() == train_ids["project id"].tolist()
assert not {"pledged", "funded percentage", "backers", "updates", "comments", "status"}.intersection(X_train.columns)

print(f"Training rows: {len(X_train):,}, success rate: {y_train.mean():.3%}")
print("Historical V3 scores are not used to select models in this notebook.")

Training rows: 29,288, success rate: 54.736%
Historical V3 scores are not used to select models in this notebook.


## 2. Field-availability and location diagnostics

`funded date` is **treated as an end/deadline timestamp by existing code**, but that is still an unverified source assumption. We inspect whether its implied launch time looks plausible; that does **not prove the column represents the deadline**. The two-letter state code check is only a *location-string heuristic* and must not be used as a country filter.

In [3]:
end = pd.to_datetime(X_train["funded date"], format="mixed", errors="coerce", utc=True)
launch = end - pd.to_timedelta(X_train["duration"], unit="D")
date_audit = pd.Series({
    "rows": len(X_train),
    "end_missing_or_unparsed": int(end.isna().sum()),
    "launch_missing_or_unparsed": int(launch.isna().sum()),
    "nonpositive_duration": int((X_train["duration"] <= 0).sum()),
    "earliest_implied_launch": str(launch.min()),
    "latest_implied_launch": str(launch.max()),
    "median_duration_days": float(X_train["duration"].median()),
})
display(date_audit.to_frame("value"))
display(launch.dt.year.value_counts(dropna=False).sort_index().rename("implied launches by year").to_frame())

US_STATES = set("AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split())
state_suffix = X_train["location"].astype("string").str.extract(r",\s*([A-Za-z]{2})\s*$", expand=False).str.upper()
location_summary = pd.Series({
    "missing_location": int(X_train["location"].isna().sum()),
    "has_two_letter_suffix": int(state_suffix.notna().sum()),
    "has_US_state_or_DC_suffix": int(state_suffix.isin(US_STATES).sum()),
    "no_confirmed_country_field": True,
})
display(location_summary.to_frame("value"))
display(state_suffix.value_counts(dropna=False).head(15).rename("rows").to_frame())
print("CA/team decision needed: follow U.S.-only milestone or retain current population?")
print("State suffixes are NOT sufficient proof of country registration.")

,value
rows,29288
end_missing_or_unparsed,0
launch_missing_or_unparsed,0
nonpositive_duration,0
earliest_implied_launch,2009-04-25 15:38:23+00:00
latest_implied_launch,2012-05-29 15:57:36+00:00
median_duration_days,32.0


,implied launches by year
2009,707
2010,5791
2011,14681
2012,8109


,value
missing_location,887
has_two_letter_suffix,26945
has_US_state_or_DC_suffix,26945
no_confirmed_country_field,True


,rows
location,
CA,5772
NY,4627
<NA>,2343
TX,1305
IL,1214
FL,962
WA,882
MA,852
PA,849


CA/team decision needed: follow U.S.-only milestone or retain current population?
State suffixes are NOT sufficient proof of country registration.


## 3. Confirm feature toggles without changing the teammate's feature generator

`FeatureBuilder(exclude_renamed=True)` drops exactly one candidate feature at the model-builder stage. Other sensitivity checks can omit all title/setup flags or the launch-timing block. They do not prove other scraped features were available at launch.

In [4]:
original = FeatureBuilder().fit_transform(X_train.iloc[:40])
conservative = FeatureBuilder(exclude_renamed=True).fit_transform(X_train.iloc[:40])
assert "renamed" in original.columns
assert "renamed" not in conservative.columns
assert set(original.columns) - set(conservative.columns) == {"renamed"}
print("Default engineered columns:", len(original.columns))
print("Without renamed:", len(conservative.columns))
print("Shared utils/features.py has not been changed.")

Default engineered columns: 35
Without renamed: 34
Shared utils/features.py has not been changed.


## 4. Training-only sensitivity comparisons (3-fold CV)

A compact **diagnostic**, not a final leaderboard. Use the **same splits and model hyperparameters** for every feature scenario. All learned transformations, including target encoding, are fit inside each training fold. Three folds and a smaller HGB iteration budget keep the audit manageable. These estimates should **not** be directly compared numerically with V3's five-fold models that use different HGB settings.

Turn off `RUN_CV` if you only need the metadata diagnostics. The notebook then skips model fits. Running the fits can take several minutes.

In [5]:
RUN_CV = True
SCENARIOS = {
    "current_features_target": dict(categorical_strategy="target"),
    "exclude_renamed": dict(categorical_strategy="target", exclude_renamed=True),
    "exclude_title_and_setup_flags": dict(categorical_strategy="target", include_flags=False),
    "exclude_launch_timing": dict(categorical_strategy="target", include_timing=False, exclude_renamed=True),
    "onehot_without_renamed": dict(categorical_strategy="onehot", exclude_renamed=True),
}
MODELS = {
    "logistic": LogisticRegression(max_iter=1500),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.06, random_state=SEED
    ),
}
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SCORING = ["roc_auc", "accuracy", "balanced_accuracy", "f1", "average_precision"]

records = []
if RUN_CV:
    for scenario, kwargs in SCENARIOS.items():
        for name, model in MODELS.items():
            pipe = make_pipeline(model, **kwargs)
            res = cross_validate(
                pipe, X_train, y_train, cv=CV, scoring=SCORING,
                n_jobs=1, error_score="raise"
            )
            record = {"scenario": scenario, "model": name}
            record.update({m: float(res[f"test_{m}"].mean()) for m in SCORING})
            record["auc_fold_std"] = float(res["test_roc_auc"].std())
            records.append(record)
            print(f"Completed {scenario} / {name}")

sensitivity_results = pd.DataFrame(records)
if not sensitivity_results.empty:
    display(sensitivity_results.sort_values(["model", "roc_auc"], ascending=[True, False]).round(4))

Completed current_features_target / logistic
Completed current_features_target / hist_gradient_boosting
Completed exclude_renamed / logistic
Completed exclude_renamed / hist_gradient_boosting
Completed exclude_title_and_setup_flags / logistic
Completed exclude_title_and_setup_flags / hist_gradient_boosting
Completed exclude_launch_timing / logistic
Completed exclude_launch_timing / hist_gradient_boosting
Completed onehot_without_renamed / logistic
Completed onehot_without_renamed / hist_gradient_boosting


,scenario,model,roc_auc,accuracy,balanced_accuracy,f1,average_precision,auc_fold_std
1,current_features_target,hist_gradient_boosting,0.7671,0.7062,0.6967,0.7481,0.7744,0.0055
3,exclude_renamed,hist_gradient_boosting,0.7665,0.7065,0.6970,0.7483,0.7737,0.0054
9,onehot_without_renamed,hist_gradient_boosting,0.7662,0.7056,0.6959,0.7481,0.7731,0.0035
7,exclude_launch_timing,hist_gradient_boosting,0.7642,0.7057,0.6960,0.7480,0.7713,0.0051
5,exclude_title_and_setup_flags,hist_gradient_boosting,0.7641,0.7055,0.6961,0.7473,0.7712,0.0054
8,onehot_without_renamed,logistic,0.7661,0.7070,0.6998,0.7435,0.7696,0.0049
0,current_features_target,logistic,0.7647,0.7055,0.6982,0.7422,0.7686,0.0052
2,exclude_renamed,logistic,0.7647,0.7053,0.6981,0.7422,0.7685,0.0051
6,exclude_launch_timing,logistic,0.7630,0.7015,0.6942,0.7388,0.7661,0.0054
4,exclude_title_and_setup_flags,logistic,0.7607,0.7015,0.6940,0.7393,0.7650,0.0057


## 5. Chronological robustness diagnostic (within training rows only)

This is **not the official test set**. The last 20% of training-row implied launch dates act as a future validation window; **earlier training campaigns must have ended before the validation window begins** to ensure their observed outcomes would have been available. Unlike random CV, this measures historical change over time. The chronological interpretation is **conditional on verifying `funded date` as the true campaign deadline**. Results are provisional until then.

In [6]:
RUN_TEMPORAL = True
if RUN_TEMPORAL:
    valid_dates = launch.notna() & end.notna()
    chronological_launch = launch.loc[valid_dates].sort_values()
    if len(chronological_launch) < 100:
        raise ValueError("Insufficient valid dates for a chronological diagnostic")
    cutoff = chronological_launch.iloc[int(len(chronological_launch) * 0.80)]
    previous_campaigns = valid_dates & (end < cutoff)
    future_campaigns = valid_dates & (launch >= cutoff)

    earlier_X = X_train.loc[previous_campaigns]
    earlier_y = y_train.loc[previous_campaigns]
    later_X = X_train.loc[future_campaigns]
    later_y = y_train.loc[future_campaigns]
    print("Implied validation launch begins:", cutoff)
    print("Completed-before-cutoff training rows:", len(earlier_X))
    print("Later-launch validation rows:", len(later_X))
    assert bool((end.loc[previous_campaigns] < cutoff).all())
    assert not set(earlier_X["project id"]).intersection(later_X["project id"])
    if len(earlier_X) < 100 or len(later_X) < 100 or earlier_y.nunique() != 2 or later_y.nunique() != 2:
        print("Skipping: insufficient rows or both classes not represented.")
    else:
        temporal_results = []
        for scenario, kwargs in {
            "current_features_target": SCENARIOS["current_features_target"],
            "exclude_renamed": SCENARIOS["exclude_renamed"],
            "exclude_launch_timing": SCENARIOS["exclude_launch_timing"],
        }.items():
            model = make_pipeline(LogisticRegression(max_iter=1500), **kwargs)
            model.fit(earlier_X, earlier_y)
            prob = model.predict_proba(later_X)[:, 1]
            predictions = (prob >= 0.5).astype(int)
            temporal_results.append({
                "scenario": scenario,
                "train_n": len(earlier_X),
                "future_n": len(later_X),
                "roc_auc": roc_auc_score(later_y, prob),
                "average_precision": average_precision_score(later_y, prob),
                "accuracy": accuracy_score(later_y, predictions),
                "balanced_accuracy": balanced_accuracy_score(later_y, predictions),
            })
        display(pd.DataFrame(temporal_results).round(4))

Implied validation launch begins: 2012-02-10 15:53:21+00:00
Completed-before-cutoff training rows: 21351
Later-launch validation rows: 5858


,scenario,train_n,future_n,roc_auc,average_precision,accuracy,balanced_accuracy
0,current_features_target,21351,5858,0.7904,0.7921,0.7235,0.7155
1,exclude_renamed,21351,5858,0.7905,0.7921,0.7245,0.7167
2,exclude_launch_timing,21351,5858,0.7888,0.7893,0.7183,0.7116


## 6. Validation findings and handoff to tuning

**Computational diagnostics complete:** 29,288 official training campaigns, 8/8 regression tests passed, 10/10 three-fold sensitivity comparisons and 3/3 chronological comparisons completed. The canonical held-out test set was not used for model selection.

### Numerical findings

| Feature configuration | Logistic ROC-AUC | HGB ROC-AUC |
|---|---:|---:|
| Current features with target encoding | 0.7647 | 0.7671 |
| Exclude `renamed` | 0.7647 | 0.7665 |
| Exclude title/setup flags | 0.7607 | 0.7641 |
| Exclude launch timing **and** `renamed` | 0.7630 | 0.7642 |
| One-hot encoding without `renamed` | 0.7661 | 0.7662 |

- Excluding `renamed` changes HGB ROC-AUC by **−0.0006** and does not change Logistic ROC-AUC to four decimal places. A conservative exclusion is therefore supported.
- Relative to excluding `renamed` alone, further excluding timing changes ROC-AUC by **−0.0017 (Logistic)** and **−0.0023 (HGB)**.
- One-hot encoding without `renamed` slightly improves Logistic ROC-AUC over target encoding without `renamed` (**0.7661 vs 0.7647**), while HGB slightly favors target encoding (**0.7665 vs 0.7662**). These differences are small relative to observed CV fold variability; no statistical significance claim is made.

### Chronological diagnostic (Logistic Regression only)

**Implied cutoff:** 2012-02-10 15:53:21 UTC. **Earlier completed campaigns:** 21,351. **Later validation campaigns:** 5,858. Another 2,079 records met neither strict criterion and were not used in this comparison.

| Configuration | ROC-AUC | Average precision | Accuracy | Balanced accuracy |
|---|---:|---:|---:|---:|
| Current features | 0.7904 | 0.7921 | 0.7235 | 0.7155 |
| Exclude `renamed` | **0.7905** | **0.7921** | **0.7245** | **0.7167** |
| Exclude timing and `renamed` | 0.7888 | 0.7893 | 0.7183 | 0.7116 |

These are encouraging diagnostics but **not definitive prospective validation**: campaign ordering and label-availability rules depend on the unverified meaning of `funded date`, and historical availability of scraped features also remains uncertain.

### Handoff decisions

- [x] Validate the official training-only data and feature-exclusion mechanism.
- [x] Pass the eight encoder/pipeline regression tests.
- [x] Complete and interpret all sensitivity and chronological comparisons.
- [ ] Confirm the original source definition of `funded date`, and whether launch timing derivatives are permissible.
- [ ] Confirm that `name`, `reward levels`, and other scraped values were observable at prediction time.
- [ ] Confirm U.S.-only versus broader population with the Challenge Advisor.
- [ ] Obtain code-owner review of the revised `utils/preprocessing.py`.
- [ ] Approve the tuning feature configuration and encoding strategy.

**Provisional plan:** exclude `renamed`; retain target-encoded HGB and one-hot Logistic Regression as tuning candidates. If source evidence does not support launch timing, use `include_timing=False` and review other potentially edited feature groups. **Do not open the official test set** during this decision process.

Full metrics, interpretation, and follow-up decisions are documented in `docs/model_validation.md`.


In [7]:
print(
    "Sensitivity comparisons:",
    len(sensitivity_results)
    if "sensitivity_results" in globals()
    else "Not available"
)

print(
    "Temporal comparisons:",
    len(temporal_results)
    if "temporal_results" in globals()
    else "Not available"
)

Sensitivity comparisons: 10
Temporal comparisons: 3


In [8]:
sensitivity_results = pd.DataFrame(sensitivity_results)
temporal_results = pd.DataFrame(temporal_results)

display(sensitivity_results.round(4))
display(temporal_results.round(4))

,scenario,model,roc_auc,accuracy,balanced_accuracy,f1,average_precision,auc_fold_std
0,current_features_target,logistic,0.7647,0.7055,0.6982,0.7422,0.7686,0.0052
1,current_features_target,hist_gradient_boosting,0.7671,0.7062,0.6967,0.7481,0.7744,0.0055
2,exclude_renamed,logistic,0.7647,0.7053,0.6981,0.7422,0.7685,0.0051
3,exclude_renamed,hist_gradient_boosting,0.7665,0.7065,0.6970,0.7483,0.7737,0.0054
4,exclude_title_and_setup_flags,logistic,0.7607,0.7015,0.6940,0.7393,0.7650,0.0057
5,exclude_title_and_setup_flags,hist_gradient_boosting,0.7641,0.7055,0.6961,0.7473,0.7712,0.0054
6,exclude_launch_timing,logistic,0.7630,0.7015,0.6942,0.7388,0.7661,0.0054
7,exclude_launch_timing,hist_gradient_boosting,0.7642,0.7057,0.6960,0.7480,0.7713,0.0051
8,onehot_without_renamed,logistic,0.7661,0.7070,0.6998,0.7435,0.7696,0.0049
9,onehot_without_renamed,hist_gradient_boosting,0.7662,0.7056,0.6959,0.7481,0.7731,0.0035


,scenario,train_n,future_n,roc_auc,average_precision,accuracy,balanced_accuracy
0,current_features_target,21351,5858,0.7904,0.7921,0.7235,0.7155
1,exclude_renamed,21351,5858,0.7905,0.7921,0.7245,0.7167
2,exclude_launch_timing,21351,5858,0.7888,0.7893,0.7183,0.7116
